In [3]:
import pandas as pd
import re

In [4]:
df = pd.read_excel('Диалоги пилотов.xlsx')

Выполним грубую разметку на штатные и экстренные ситуации

In [5]:
# список слов-маркеров
emergency_markers = [
    'emergency', 'mayday', 'pan pan', 'failed', 'failure', 
    'fire', 'stop immediately', 'going around', 'belly landing', 
    'structural failure', 'engine out', 'on fire', 'warning light',
    'priority landing', 'emergency descent', 'evacuation'
]

def mark_risk(text):
    text = str(text).lower()
    for word in emergency_markers:
        if word in text:
            return 1
    return 0

# Применяем функцию ко всем диалогам
df['Label'] = df['Dialogs Pilot/Controller'].apply(mark_risk)

# Сохраняем размеченный файл
df.to_excel('Диалоги_готовые_для_ML.xlsx', index=False)
print(df['Label'].value_counts())

Label
0    25
1    15
Name: count, dtype: int64


Попробуем так же сделать груюу. классификацию

In [6]:
def classify_problem(text):
    text = str(text).lower()
    if 'engine' in text or 'bird' in text or 'surge' in text:
        return 'Engine'
    elif 'ice' in text or 'turbulence' in text or 'wind shear' in text:
        return 'Weather'
    elif 'position' in text or 'radar' in text or 'vector' in text:
        return 'Navigation'
    elif 'hydraulic' in text or 'gear' in text or 'brake' in text:
        return 'Mechanical'
    else:
        return 'Other'

In [7]:
df['Class'] = df['Dialogs Pilot/Controller'].apply(classify_problem)

Как мы можем заметить, в результате выполнения данного скрипта получился очень урезанный список экстренных ситуаций. Это отражает поверхностность такого подхода. Попробуем использовать более сложные механизмы

In [8]:
def mark_risk_advanced(text):
    text = str(text).lower()
    
    # проверка по основным сигналам
    critical_words = ['emergency', 'mayday', 'pan pan', 'fire', 'on fire', 
                      'stop immediately', 'belly landing', 'evacuation']
    for word in critical_words:
        if word in text:
            return 1
    
    # проверка на фраз с отказом
    failure_patterns = [
        r'engine (failed|failure|out)', 
        r'hydraulic (pressure dropping|failure)',
        r'gear cannot be deployed',
        r'structural failure',
        r'warning light (is on|keeps coming on)',
        r'transponder (inoperative|appears inoperative)',
        r'alternator malfunctioned',
        r'electrical problem'
    ]
    for pattern in failure_patterns:
        if re.search(pattern, text):
            return 1
    
    # запросы на изменение плана
    request_patterns = [
        r'request (priority landing|emergency descent|radar vectors|return)',
        r'need to (descend immediately|dump fuel)',
        r'we have a problem',
        r'not sure of our position'
    ]
    for pattern in request_patterns:
        if re.search(pattern, text):
            return 1
    
    # метеоопасности 
    weather_patterns = [
        r'(severe|heavy) (icing|turbulence)',
        r'ice build up',
        r'wind shear',
        r'chunks of ice'
    ]
    for pattern in weather_patterns:
        if re.search(pattern, text):
            return 1
    
    return 0

In [9]:
# применение скрипта
df['Label'] = df['Dialogs Pilot/Controller'].apply(mark_risk_advanced)
print(df['Label'].value_counts())


Label
0    21
1    19
Name: count, dtype: int64


В итоге данный способ показал результат хуже предыдущего, из чего можно сделать вывод, что для начала стоит начать с ручной разметки